In [1]:
import joblib
import numpy as np
import pandas as pd

In [2]:
# Cargar el modelo y los scalers
loaded_model = joblib.load('model.pkl')
loaded_scaler_X = joblib.load('scaler_X.pkl')
loaded_scaler_y = joblib.load('scaler_y.pkl')

In [3]:
# Las columnas de entrada en el orden correcto (sin 'charges')
feature_cols = ['sex', 'smoker', 'region_northeast', 'region_northwest', 'region_southeast', 'region_southwest', 'age', 'bmi', 'children']

def predict_price(
    sex: float,
    smoker: float,
    region_northeast: float,
    region_northwest: float,
    region_southeast: float,
    region_southwest: float,
    age: float,
    bmi: float,
    children: float
) -> float:
    """
    Predice el costo del seguro basado en las características proporcionadas.

    Args:
        sex (float): 0.0 para femenino, 1.0 para masculino.
        smoker (float): 0.0 para no fumador, 1.0 para fumador.
        region_northeast (float): 1.0 si es noreste, 0.0 en otro caso.
        region_northwest (float): 1.0 si es noroeste, 0.0 en otro caso.
        region_southeast (float): 1.0 si es sureste, 0.0 en otro caso.
        region_southwest (float): 1.0 si es suroeste, 0.0 en otro caso.
        age (float): Edad del asegurado.
        bmi (float): Índice de Masa Corporal del asegurado.
        children (float): Número de hijos a cargo del asegurado.

    Returns:
        float: El costo predicho del seguro.
    """
    # Crear un array con los valores de entrada en el orden correcto
    input_data = np.array([
        sex,
        smoker,
        region_northeast,
        region_northwest,
        region_southeast,
        region_southwest,
        age,
        bmi,
        children
    ]).reshape(1, -1)

    # Escalar las características de entrada usando el scaler_X cargado
    input_scaled = loaded_scaler_X.transform(input_data)

    # Realizar la predicción con el modelo cargado
    predicted_charge_scaled = loaded_model.predict(input_scaled)

    # Invertir la escala de la predicción usando el scaler_y cargado
    predicted_charge = loaded_scaler_y.inverse_transform(predicted_charge_scaled.reshape(-1, 1))

    return predicted_charge[0][0]

# Ejemplo de uso de la función:
# Asumiendo un hombre, no fumador, en el suroeste, de 30 años, con BMI de 25 y 1 hijo.
# sex=1.0 (male), smoker=0.0 (no), region_northeast=0.0, region_northwest=0.0,
# region_southeast=0.0, region_southwest=1.0, age=30.0, bmi=25.0, children=1.0

sample_prediction = predict_price(1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 30.0, 25.0, 1.0)
print(f"El costo de seguro predicho para el ejemplo es: ${sample_prediction:.2f}")

# Otro ejemplo: mujer, fumadora, noreste, 45 años, BMI 30, 2 hijos
sample_prediction_2 = predict_price(0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 45.0, 30.0, 2.0)
print(f"El costo de seguro predicho para el segundo ejemplo es: ${sample_prediction_2:.2f}")

El costo de seguro predicho para el ejemplo es: $5079.86
El costo de seguro predicho para el segundo ejemplo es: $25019.64


In [4]:
def predict_price_by_region(
    sex: float,
    smoker: float,
    region: str, # 'northeast', 'northwest', 'southeast', 'southwest'
    age: float,
    bmi: float,
    children: float
) -> float:
    """
    Predice el costo del seguro basado en las características proporcionadas,
    recibiendo la región como una cadena de texto.

    Args:
        sex (float): 0.0 para femenino, 1.0 para masculino.
        smoker (float): 0.0 para no fumador, 1.0 para fumador.
        region (str): La región del asegurado ('northeast', 'northwest', 'southeast', 'southwest').
        age (float): Edad del asegurado.
        bmi (float): Índice de Masa Corporal del asegurado.
        children (float): Número de hijos a cargo del asegurado.

    Returns:
        float: El costo predicho del seguro.
    """
    # Inicializar todas las regiones a 0.0
    region_northeast = 0.0
    region_northwest = 0.0
    region_southeast = 0.0
    region_southwest = 0.0

    # Asignar 1.0 a la región correspondiente
    if region == 'northeast':
        region_northeast = 1.0
    elif region == 'northwest':
        region_northwest = 1.0
    elif region == 'southeast':
        region_southeast = 1.0
    elif region == 'southwest':
        region_southwest = 1.0
    else:
        raise ValueError("Región no válida. Debe ser 'northeast', 'northwest', 'southeast' o 'southwest'.")

    # Llamar a la función predict_price original con los valores de región correctos
    return predict_price(
        sex=sex,
        smoker=smoker,
        region_northeast=region_northeast,
        region_northwest=region_northwest,
        region_southeast=region_southeast,
        region_southwest=region_southwest,
        age=age,
        bmi=bmi,
        children=children
    )

# Ejemplo de uso de la nueva función:
# Asumiendo un hombre, no fumador, en el suroeste, de 30 años, con BMI de 25 y 1 hijo.
# sex=1.0 (male), smoker=0.0 (no), region='southwest', age=30.0, bmi=25.0, children=1.0

sample_prediction_region = predict_price_by_region(1.0, 0.0, 'southwest', 30.0, 25.0, 1.0)
print(f"El costo de seguro predicho (por región) para el ejemplo es: ${sample_prediction_region:.2f}")

# Otro ejemplo: mujer, fumadora, noreste, 45 años, BMI 30, 2 hijos
sample_prediction_region_2 = predict_price_by_region(0.0, 1.0, 'northeast', 45.0, 30.0, 2.0)
print(f"El costo de seguro predicho (por región) para el segundo ejemplo es: ${sample_prediction_region_2:.2f}")

El costo de seguro predicho (por región) para el ejemplo es: $5079.86
El costo de seguro predicho (por región) para el segundo ejemplo es: $25019.64


In [5]:
import gradio as gr

def gradio_predict_price(sex, smoker, region, age, bmi, children):
    # Convertir 'sex' a float (0.0 para Femenino, 1.0 para Masculino)
    sex_val = 1.0 if sex == "Masculino" else 0.0
    # Convertir 'smoker' a float (0.0 para No, 1.0 para Sí)
    smoker_val = 1.0 if smoker == "Sí" else 0.0

    try:
        prediction = predict_price_by_region(
            sex=sex_val,
            smoker=smoker_val,
            region=region.lower(), # Asegurarse de que la región esté en minúsculas
            age=float(age),
            bmi=float(bmi),
            children=float(children)
        )
        return f"El costo de seguro predicho es: ${prediction:.2f}"
    except ValueError as e:
        return f"Error: {e}"

# Definir los componentes de entrada para Gradio
inputs = [
    gr.Radio(["Femenino", "Masculino"], label="Sexo"),
    gr.Radio(["No", "Sí"], label="Fumador"),
    gr.Dropdown(['northeast', 'northwest', 'southeast', 'southwest'], label="Región"),
    gr.Number(label="Edad"),
    gr.Number(label="BMI"),
    gr.Number(label="Número de hijos")
]

# Crear la interfaz Gradio
iface = gr.Interface(
    fn=gradio_predict_price,
    inputs=inputs,
    outputs="text",
    title="Predicción del Costo del Seguro Médico",
    description="Introduce las características del asegurado para predecir el costo del seguro."
)

# Lanzar la aplicación Gradio
iface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6cf8f8bb464637e4f0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6cf8f8bb464637e4f0.gradio.live
